In [20]:
! pip install groq -q

In [21]:
!pip install google-generativeai

In [22]:
!pip install -q langchain-google-genai

Install wikipedia package and download wikipedia article

In [23]:
!pip install wikipedia-api -q

In [24]:
import wikipediaapi
try:
  wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rag-project'
  )

  topics=["quantum computing",
       "Artificial intelligence",
       "machine learning"]
  for topic in topics:
   page=wiki.page(topic)
   print("Title",page.title)
   print("Length",len(page.text))
   print("-"*30)
except Exception as e:
   print("error occur:",e)

Title quantum computing
Length 66420
------------------------------
Title Artificial intelligence
Length 83965
------------------------------
Title machine learning
Length 58125
------------------------------


save each article automatically


In [25]:
!pip install reportlab

In [26]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

for topic in topics:
    page = wiki.page(topic)

    if page.exists():
        filename = topic.replace(" ", "_") + ".pdf"

        pdf = SimpleDocTemplate(filename)
        styles = getSampleStyleSheet()

        content = [Paragraph(page.text, styles['BodyText'])]
        pdf.build(content)

        print(f"Saved: {filename}")

Saved: quantum_computing.pdf
Saved: Artificial_intelligence.pdf
Saved: machine_learning.pdf


Ingestion notebook:loads documents,chunks,embeds,stores in chromaDB

In [27]:
!pip install -U langchain-text-splitters
!pip install -U langchain-community
!pip install -U langchain-google-genai
!pip install -U chromadb

  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.2.4
    Uninstalling langchain-text-splitters-0.2.4:
      Successfully uninstalled langchain-text-splitters-0.2.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.2.17 requires langchain-core<0.3.0,>=0.2.43, but you have langchain-core 1.4.1 which is incompatible.
langchain 0.2.17 requires langchain-text-splitters<0.3.0,>=0.2.0, but you have langchain-text-splitters 1.1.2 which is incompatible.
langchain 0.2.17 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.8.9 which is incompatible.
langchain 0.2.17 requires tenacity!=8.4.0,<9.0.0,>=8.1.0, but you have tenacity 9.1.4 which is incom

In [28]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

In [29]:
!pip install -q langchain pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wikipedia-api 0.15.0 requires tenacity==9.1.4, but you have tenacity 8.5.0 which is incompatible.
langchain-google-genai 4.2.4 requires langchain-core<2.0.0,>=1.3.2, but you have langchain-core 0.2.43 which is incompatible.
ragas 0.1.21 requires langchain-community<0.3, but you have langchain-community 0.4.2 which is incompatible.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
instructor 1.15.1 requires openai<3.0.0,>=2.0.0, but you have openai 1.109.1 which is incompatible.
langchain-community 0.4.2 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.
google-a

local embedding instead of gemini


In [30]:
!pip install -q sentence-transformers

In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings



try:
    pdf_files = [
        "quantum_computing.pdf",
        "Artificial_intelligence.pdf",
        "machine_learning.pdf"
    ]

    all_docs = []

    for pdf in pdf_files:
        loader = PyPDFLoader(pdf)
        docs = loader.load()
        all_docs.extend(docs)

    print(f"Total pages loaded: {len(all_docs)}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(all_docs)

    print(f"Total chunks created: {len(chunks)}")

    embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
    vectorstore = Chroma.from_documents(
        chunks,
        embeddings,
        persist_directory="./chroma_db"
    )

    print(f"Successfully stored {len(chunks)} chunks in ChromaDB")

except Exception as e:
    print("Error:", e)

Total pages loaded: 38
Total chunks created: 437


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully stored 437 chunks in ChromaDB


In [32]:
!pip install -U langchain-google-genai google-generativeai

  Using cached langchain_core-1.4.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached langsmith-0.8.9-py3-none-any.whl.metadata (15 kB)
Using cached langchain_core-1.4.1-py3-none-any.whl (549 kB)
Using cached langsmith-0.8.9-py3-none-any.whl (403 kB)
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.147
    Uninstalling langsmith-0.1.147:
      Successfully uninstalled langsmith-0.1.147
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.2.43
    Uninstalling langchain-core-0.2.43:
      Successfully uninstalled langchain-core-0.2.43
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.1.25 requires langchain-core<0.3.0,>=0.2.40, but you have langchain-core 1.4.1 which is incompatible.
langchain-text-splitters 0.2.4 requires langchain-core<0.3.0,>=0.2.38, but you have langchain-co

In [33]:
!pip show langchain-google-genai
!pip show google-generativeai

Name: langchain-google-genai
Version: 4.2.4
Summary: An integration package connecting Google's genai package and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/google
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: filetype, google-genai, langchain-core, pydantic
Required-by: 
Name: google-generativeai
Version: 0.8.6
Summary: Google Generative AI High level API client library and tools.
Home-page: https://github.com/google/generative-ai-python
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: google-ai-generativelanguage, google-api-core, google-api-python-client, google-auth, protobuf, pydantic, tqdm, typing-extensions
Required-by: 


In [34]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("geminikey"))

for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(model.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


Groq client.Import the groq library to access LLM MODEL

In [35]:
from groq import Groq

create a groq client using the API key

In [36]:
client = Groq(api_key=userdata.get("groqkey"))

BUILD THE QUERY NOTEBOOK.Take a question,retrieve the 5 most similsr chunks,pass them to to groq with a clear prompt,and display answer


In [37]:
from groq import Groq

client = Groq(api_key=userdata.get("groqkey"))

try:
    question = input("Ask a question: ")

    docs_found = vectorstore.similarity_search(
        question,
        k=5
    )

    context = "\n\n".join(
        [doc.page_content for doc in docs_found]
    )

    prompt = f"""
Answer only from the context.

Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role":"user","content":prompt}
        ]
    )

    print(response.choices[0].message.content)

except Exception as e:
    print("Error:", e)

Ask a question: what is quantum computing?
A quantum computer is a real or theoretical computer that exploits quantum phenomena like superposition and entanglement in an essential way. It is widely believed that a quantum computer could perform some calculations exponentially faster than any classical computer.


CONNECT TO LangGraph OR n8n path- Wire everything together into a proper pipeline with proper error handling

In [38]:
!pip install langgraph -q

In [39]:
from typing import TypedDict
from langgraph.graph import StateGraph

class State(TypedDict):
    question: str
    context: str
    answer: str

def retrieve(state):
    docs = vectorstore.similarity_search(
        state["question"],
        k=5
    )

    context = "\n".join(
        [d.page_content for d in docs]
    )

    return {"context": context}

def generate(state):
    prompt = f"""
Context:
{state['context']}

Question:
{state['question']}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role":"user","content":prompt}
        ]
    )

    return {
        "answer":
        response.choices[0].message.content
    }

graph = StateGraph(State)

graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")

app = graph.compile()

result = app.invoke(
    {"question":"What is quantum computing?"}
)

print(result["answer"])

Quantum computing is a type of computing that exploits quantum phenomena like superposition and entanglement to perform calculations. It is believed to have the potential to perform some calculations exponentially faster than classical computers, making it useful for tasks such as breaking certain encryption schemes and aiding physicists in physical simulations. Quantum computing uses mathematical models like linear algebra, complex numbers, vectors, and matrices to describe and program quantum systems, with the goal of composing operations to achieve a useful result.


Evaluate with RAGAS-Manually write 20 questions about ypur documents.run RAGAS and note the scores.Find 3 questions where the answer was wrong and understand why

In [40]:
!pip install ragas -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 4.2.4 requires langchain-core<2.0.0,>=1.3.2, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langgraph 1.2.4 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.43 which is incompatible.


In [41]:
questions=[
"What is quantum computing?",
"What is a qubit?",
"How is a qubit different from a classical bit?",
"What is quantum superposition?",
"What is quantum entanglement?",
"What are the applications of quantum computing?",
"What are the limitations of quantum computers?",
"What is Artificial Intelligence?",
"What are the main types of AI?",
"What is the difference between AI and Machine Learning?",
"What are some real-world applications of AI?",
"What are the advantages of Artificial Intelligence?",
"What are the challenges of AI?",
"What is Machine Learning?",
"What are the types of Machine Learning?",
"What is supervised learning?",
"What is unsupervised learning?",
"What is reinforcement learning?",
"What is overfitting in Machine Learning?",
"What is the role of training data in Machine Learning?"]

In [42]:
!pip install ragas datasets -q

In [43]:
!pip uninstall -y ragas langchain langchain-community langchain-core

Found existing installation: ragas 0.1.21
Uninstalling ragas-0.1.21:
  Successfully uninstalled ragas-0.1.21
Found existing installation: langchain 0.2.17
Uninstalling langchain-0.2.17:
  Successfully uninstalled langchain-0.2.17
Found existing installation: langchain-community 0.2.19
Uninstalling langchain-community-0.2.19:
  Successfully uninstalled langchain-community-0.2.19
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43


In [44]:
!pip install -U ragas langchain langchain-community langchain-core

  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached langchain-1.3.4-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_core-1.4.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached langsmith-0.8.9-py3-none-any.whl.metadata (15 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.2.2-py3-none-any.whl.metadata (3.1 kB)
Using cached ragas-0.4.3-py3-none-any.whl (466 kB)
Using cached langchain-1.3.4-py3-none-any.whl (125 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_core-1.4.1-py3-none-any.whl (549 kB)
Using cached langsmith-0.8.9-py3-none-any.whl (403 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.3 MB/s eta 

In [45]:
!pip install google-cloud-aiplatform

In [47]:
!pip show ragas
!pip show langchain-community

Name: ragas
Version: 0.4.3
Summary: Evaluation framework for RAG and LLM applications
Home-page: https://github.com/vibrantlabsai/ragas
Author: 
Author-email: 
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by

In [48]:
!pip uninstall -y ragas
!pip install ragas==0.1.21

Found existing installation: ragas 0.4.3
Uninstalling ragas-0.4.3:
  Successfully uninstalled ragas-0.4.3
  Using cached ragas-0.1.21-py3-none-any.whl.metadata (5.3 kB)
  Using cached langchain-0.2.17-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.2.19-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.2.1-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.2.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.16-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_openai-1.1.15-py3-none-any.whl.metadata (3.1 kB)
  Using cached langc

In [51]:

!pip install langchain-google-vertexai

  Using cached langchain_core-1.4.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached langsmith-0.8.9-py3-none-any.whl.metadata (15 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 8.3 MB/s eta 0:00:00
Using cached langchain_core-1.4.1-py3-none-any.whl (549 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.3 MB/s eta 0:00:00
Using cached langsmith-0.8.9-py3-none-any.whl (403 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/375.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [52]:

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy

questions_list = []
answers = []
contexts = []


for question in questions:

    docs_found = vectorstore.similarity_search(
        question,
        k=5
    )

    context = [doc.page_content for doc in docs_found]

    prompt = f"""
Answer only from the context.

Context:
{' '.join(context)}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    questions_list.append(question)
    answers.append(answer)
    contexts.append(context)

dataset = Dataset.from_dict(
    {
        "question": questions_list,
        "answer": answers,
        "contexts": contexts
    }
)

result = evaluate(
    dataset,
    metrics=[
        answer_relevancy
    ]
)

print(result)

TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases